# 05 — Timezones and sessions

This notebook starts from the bug that motivated the library's whole timezone policy, then
shows the tool that closes it: `session_of`.

The rule fits in three lines:

1. **Naive means "already in the right frame."** The date part is taken literally, with no
   conversion.
2. **Aware means "an instant."** Projecting it onto a calendar day requires an explicit
   timezone.
3. **Offsets preserve wall-clock time and tzinfo.** They only touch the date part.

In [1]:
from datetime import date, datetime, timedelta, timezone
from zoneinfo import ZoneInfo

import pandas as pd

import better_calendar as bcal
from better_calendar import Calendar

## 1. The bug

The same instant. Two answers. No warning.

In [2]:
ts = pd.Timestamp("2026-07-31 23:30", tz="UTC")

print("the instant              :", ts)
print("ts.date()                :", ts.date(), f"({ts.date():%A})")
print("ts.tz_convert('Paris')   :", ts.tz_convert("Europe/Paris").date(),
      f"({ts.tz_convert('Europe/Paris').date():%A})")

the instant              : 2026-07-31 23:30:00+00:00
ts.date()                : 2026-07-31 (Friday)
ts.tz_convert('Paris')   : 2026-08-01 (Saturday)


`.date()` answers according to whichever zone the timestamp happens to be carrying. In a
pipeline where a `tz_convert` sits three functions higher, the answer changes without
anybody deciding anything — and the consequence is not cosmetic:

In [3]:
utc = Calendar("utc", tz="UTC")
paris = Calendar("paris", tz="Europe/Paris")

print("business day in UTC   ?", utc.is_bday(ts))
print("business day in Paris ?", paris.is_bday(ts))

business day in UTC   ? True
business day in Paris ? False


## 2. The fix

`session_of` makes you name the frame, and tells you back.

In [4]:
pd.DataFrame(
    [
        {"frame": name, "day": bcal.session_of(ts, tz=zone),
         "day name": bcal.session_of(ts, tz=zone).strftime("%A")}
        for name, zone in [
            ("UTC", "UTC"), ("Europe/Paris", "Europe/Paris"),
            ("Asia/Tokyo", "Asia/Tokyo"), ("America/New_York", "America/New_York"),
        ]
    ]
).set_index("frame")

,day,day name
frame,,
UTC,2026-07-31,Friday
Europe/Paris,2026-08-01,Saturday
Asia/Tokyo,2026-08-01,Saturday
America/New_York,2026-07-31,Friday


And there is **no** silent fallback. The `weekday` calendar declares no timezone:

In [5]:
try:
    bcal.session_of(ts)
except bcal.AmbiguousTimezoneError as exc:
    print(exc)

session_of needs a timezone: calendar 'weekday' declares none, so there is no frame to read an instant in. Pass tz=..., use a calendar that has one, or set better_calendar.config.default_tz. A composite calendar loses its timezone when its operands disagree, which is usually the cause.


Three ways to supply one: the argument, the calendar, or the global escape hatch.

In [6]:
print("via the argument :", bcal.session_of(ts, tz="Europe/Paris"))
print("via the calendar :", bcal.session_of(ts, cal=paris))

bcal.config.default_tz = "Europe/Paris"        # the library's only global state
print("via the default  :", bcal.session_of(ts))
bcal.config.default_tz = None

via the argument : 2026-08-01
via the calendar : 2026-08-01
via the default  : 2026-08-01


## 3. Naive versus aware

A naive `datetime` is a **label**, not an instant: its date part is read as-is, with no
conversion, even if you pass a timezone.

In [7]:
naive = datetime(2026, 7, 31, 23, 30)
aware = datetime(2026, 7, 31, 23, 30, tzinfo=timezone.utc)

print("naive, tz=Paris :", bcal.session_of(naive, tz="Europe/Paris"), " <- read literally")
print("aware, tz=Paris :", bcal.session_of(aware, tz="Europe/Paris"), " <- converted")

naive, tz=Paris : 2026-07-31  <- read literally
aware, tz=Paris : 2026-08-01  <- converted


## 4. Offsets preserve the wall clock

An offset only touches the date part. Across a daylight-saving transition that means one
business day is +23h or +25h in absolute terms — and that is **intended**.

In [8]:
PARIS = ZoneInfo("Europe/Paris")
before = datetime(2026, 3, 27, 9, 0, tzinfo=PARIS)          # Friday, before the change
after = paris.offset(before, 1)                              # -> Monday

elapsed = after.astimezone(timezone.utc) - before.astimezone(timezone.utc)
print("before      :", before)
print("+1 business :", after)
print("wall clock unchanged :", before.hour == after.hour)
print("time actually elapsed:", elapsed, "instead of 72h")

before      : 2026-03-27 09:00:00+01:00
+1 business : 2026-03-30 09:00:00+02:00
wall clock unchanged : True
time actually elapsed: 2 days, 23:00:00 instead of 72h


## 5. Sessions

A calendar day is the interval `[session_start, session_start + 24h)` expressed in the
calendar's timezone. Local midnight for most, `00:00` UTC for crypto, `17:00` New York for
FX.

In [9]:
fx = Calendar("fx", tz="America/New_York", session_start=pd.Timestamp("17:00").time())

morning = pd.Timestamp("2026-07-31 09:00", tz="America/New_York")
evening = pd.Timestamp("2026-07-31 18:00", tz="America/New_York")

print("session_start = 17:00 New York")
print(f"  {morning}  -> session {bcal.session_of(morning, cal=fx)}   (opened the previous evening)")
print(f"  {evening}  -> session {bcal.session_of(evening, cal=fx)}")

session_start = 17:00 New York
  2026-07-31 09:00:00-04:00  -> session 2026-07-30   (opened the previous evening)
  2026-07-31 18:00:00-04:00  -> session 2026-07-31


`session_bounds` gives the half-open UTC interval a day covers:

In [10]:
xpar = bcal.get("XPAR")
pd.DataFrame(
    [
        {
            "day": day,
            "start (UTC)": str(xpar.session_bounds(day)[0]),
            "end (UTC)": str(xpar.session_bounds(day)[1]),
            "length": str(xpar.session_bounds(day)[1] - xpar.session_bounds(day)[0]),
        }
        for day in ("2026-03-27", "2026-03-29", "2026-07-31", "2026-10-25")
    ]
).set_index("day")

,start (UTC),end (UTC),length
day,,,
2026-03-27,2026-03-26 23:00:00+00:00,2026-03-27 23:00:00+00:00,1 days 00:00:00
2026-03-29,2026-03-28 23:00:00+00:00,2026-03-29 22:00:00+00:00,0 days 23:00:00
2026-07-31,2026-07-30 22:00:00+00:00,2026-07-31 22:00:00+00:00,1 days 00:00:00
2026-10-25,2026-10-24 22:00:00+00:00,2026-10-25 23:00:00+00:00,1 days 01:00:00


A session really is 23 or 25 hours long around a clock change. That is not a defect to
normalise away: it is the code assuming 24 hours that this function exists to correct.

The two functions are consistent by construction — every instant falls inside the bounds of
the day `session_of` assigns it to:

In [11]:
ok = True
for calendar in (utc, paris, fx, bcal.get("crypto:24x7")):
    for hour in range(0, 24, 3):
        moment = pd.Timestamp(f"2026-07-31 {hour:02d}:17", tz="UTC")
        day = bcal.session_of(moment, cal=calendar)
        opens, closes = calendar.session_bounds(day)
        ok &= bool(opens <= moment < closes)
print("session_of / session_bounds agree over 4 calendars x 8 hours :", ok)

session_of / session_bounds agree over 4 calendars x 8 hours : True


## 6. `grid`: the mis-anchored resample

This is the classic trap. A four-hour grid built from UTC midnight cuts a Paris or Tokyo
session in the wrong places.

In [12]:
naive_pandas = pd.date_range("2026-07-31 00:00", periods=6, freq="4h", tz="UTC")
aligned = xpar.grid("2026-07-31", "2026-07-31", "4h")

pd.DataFrame(
    {
        "pandas from UTC midnight": naive_pandas.tz_convert("Europe/Paris").strftime("%m-%d %H:%M"),
        "grid anchored on the session": aligned.tz_convert("Europe/Paris").strftime("%m-%d %H:%M"),
    }
)

,pandas from UTC midnight,grid anchored on the session
0,07-31 02:00,07-31 00:00
1,07-31 06:00,07-31 04:00
2,07-31 10:00,07-31 08:00
3,07-31 14:00,07-31 12:00
4,07-31 18:00,07-31 16:00
5,07-31 22:00,07-31 20:00


The left column starts at 02:00 Paris time: the bars are two hours out of step with the
actual day. The right column starts at local midnight.

`grid` covers **sessions** only, so no bars at the weekend for an exchange, and a
continuous grid for crypto:

In [13]:
print("XPAR, Saturday + Sunday  :", len(xpar.grid("2026-08-01", "2026-08-02", "6h")), "points")
print("crypto, Saturday + Sunday:", len(bcal.get("crypto:24x7").grid("2026-08-01", "2026-08-02", "6h")), "points")

XPAR, Saturday + Sunday  : 0 points
crypto, Saturday + Sunday: 8 points


Each session is filled independently, so a clock change shortens **that** session without
shifting every later point:

In [14]:
around = xpar.grid("2026-03-27", "2026-03-31", "6h")
local = around.tz_convert("Europe/Paris")
pd.DataFrame({"UTC": around.strftime("%m-%d %H:%M"), "Paris": local.strftime("%m-%d %H:%M")})

,UTC,Paris
0,03-26 23:00,03-27 00:00
1,03-27 05:00,03-27 06:00
2,03-27 11:00,03-27 12:00
3,03-27 17:00,03-27 18:00
4,03-29 22:00,03-30 00:00
5,03-30 04:00,03-30 06:00
6,03-30 10:00,03-30 12:00
7,03-30 16:00,03-30 18:00
8,03-30 22:00,03-31 00:00
9,03-31 04:00,03-31 06:00


## 7. `at_times`: crossing days with times of day

The companion to the recurrences: generate the days, then attach the times a process
actually runs at.

In [15]:
fixings = bcal.at_times(bcal.imm_dates("2026-01-01", "2026-12-31"), ["08:00", "16:00"])
pd.DataFrame({"UTC": fixings.strftime("%Y-%m-%d %H:%M%z")})

,UTC
0,2026-03-18 08:00+0000
1,2026-03-18 16:00+0000
2,2026-06-17 08:00+0000
3,2026-06-17 16:00+0000
4,2026-09-16 08:00+0000
5,2026-09-16 16:00+0000
6,2026-12-16 08:00+0000
7,2026-12-16 16:00+0000


In [16]:
# In another timezone, with seconds.
bcal.at_times(["2026-07-31"], ["09:30:15", "17:00"], tz="Europe/Paris").strftime("%Y-%m-%d %H:%M:%S %Z").tolist()

['2026-07-31 09:30:15 CEST', '2026-07-31 17:00:00 CEST']

## 8. What is deliberately absent

No `is_open`, no `next_open`, no lunch breaks, no early closes.

An `is_open()` returning `is_bday()` would be **false** for any exchange with trading hours
— and it would be discovered the hard way. The distinction is written down as two
protocols, `DayCalendar` (satisfied today) and `SessionCalendar` (satisfied by nothing), so
it survives the next person to touch this code.

In [17]:
[name for name in ("is_open", "next_open", "next_close", "trading_minutes")
 if hasattr(bcal.get("XNYS"), name)] or "none of those methods exist"

'none of those methods exist'

## Recap

| Call | Role |
|---|---|
| `session_of(ts, cal=, tz=)` | which calendar day this instant belongs to |
| `session_bounds(day, cal=)` | the half-open UTC interval of that day |
| `cal.grid(a, b, "4h")` | grid anchored on `session_start`, not UTC midnight |
| `at_times(days, times, tz=)` | cross a recurrence with times of day |
| `config.default_tz` | the global escape hatch, off by default |

**Next:** [06 — Snapshots and overrides](06-snapshots-and-overrides.ipynb)